In [1]:
import sys
import os
import dask
import dask.dataframe as dd
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
mc_med_path = '~/mc-med/mc-med-1.0.0/data/'

## Labs

In [3]:
labs = dd.read_csv(mc_med_path + 'labs.csv', assume_missing=True, dtype={'Component_value': 'string'})
# preprocess: Convert CSN to int
labs['CSN'] = labs['CSN'].astype('Int64')
# Drop columns except CSN, Result_time, Component_value, Component_name, Component_units
labs = labs[['CSN', 'Result_time', 'Component_value', 'Component_name', 'Component_units', 'Component_result']]
# Rename 'Result_time' to 'Time'
labs = labs.rename(columns={'Result_time': 'Time'}).compute()

KeyboardInterrupt: 

In [ ]:
# Get the labs where Component_units is NaN
labs_nounits = labs[labs['Component_units'].isna()].copy()
# Add a text column: 'Component_name: Component_result'
labs_nounits['Text'] = labs_nounits['Component_name'] + ': ' + labs_nounits['Component_result']
# keep CSN, Text and Time
labs_nounits = labs_nounits[['CSN', 'Text', 'Time']]

In [31]:
# Get the labs where Component_units is not NaN
labs_units = labs[labs['Component_units'].notna()].copy()
# Add a text column: 'Component_name: Component_value Component_units'
labs_units['Text'] = labs_units['Component_name'] + ': ' + labs_units['Component_value'] + ' ' + labs_units['Component_units']
# keep CSN, Text and Time
labs_units = labs_units[['CSN', 'Text', 'Time']]
# Concatenate the two dataframes
labs = pd.concat([labs_nounits, labs_units], ignore_index=True)

In [6]:
# Store as parquet in intermediate directory (current folder)
intermediate_dir = 'intermediate'
if not os.path.exists(intermediate_dir):
    os.makedirs(intermediate_dir)
# labs.to_parquet(os.path.join(intermediate_dir, 'labs.parquet'), engine='pyarrow', index=False)


## Vitals

In [10]:
vitals = dd.read_csv(mc_med_path + 'numerics.csv', assume_missing=True)
# Preprocess: Convert CSN to int
vitals['CSN'] = vitals['CSN'].astype('Int64')
# Drop columns except CSN, Measure, Value, Time
vitals = vitals[['CSN', 'Measure', 'Value', 'Time']].compute()

In [ ]:
# Drop rows where Measure is HR or SPO2 (i.e., less frequently measured Vitals)
vitals_nofreq = vitals[~vitals['Measure'].isin(['HR', 'SPO2'])].copy()
# Convert Value column to str
vitals_nofreq['Value'] = vitals_nofreq['Value'].astype(str)
# Drop rows with any missing values in 'Value' or 'Measure'
vitals_nofreq = vitals_nofreq.dropna(subset=['Value', 'Measure'])
# Explicitly convert 'Measure' to string right before concatenation
vitals_nofreq['Measure'] = vitals_nofreq['Measure'].astype(str)
# Add text column 'Measure: Value'
vitals_nofreq['Text'] = vitals_nofreq['Measure'] + ': ' + vitals_nofreq['Value']
# keep just the CSN, Text and Time columns in vitals_nofreq
vitals_nofreq = vitals_nofreq[['CSN', 'Text', 'Time']].compute()
# Write to parquet
vitals_nofreq.to_parquet(os.path.join(intermediate_dir, 'vitals_nofreq.parquet'), engine='pyarrow', index=False)

NameError: name 'intermediate_dir' is not defined

In [ ]:
# Keep the HR and Sp02 rows
vitals_freq = vitals[vitals['Measure'].isin(['HR', 'SPO2'])].copy()
# Write to parquet
vitals_freq.to_parquet(os.path.join(intermediate_dir, 'vitals_freq.parquet'), engine='pyarrow', index=False)


In [17]:
orders = dd.read_csv(mc_med_path + 'orders.csv', assume_missing=True, dtype={'First_admin_time': 'object'})
# Preprocess: Convert CSN to int
orders['CSN'] = orders['CSN'].astype('Int64')
# Let's only get the medications
medications = orders[orders['Order_type'] == 'Medications']

In [18]:
# Drop columns except CSN, Procedure_name, First_admin_time
medications = medications[['CSN', 'Procedure_name', 'First_admin_time']]
# Rename 'First_admin_time' to 'Time', and 'Procedure_name' to 'Text'
medications = medications.rename(columns={'First_admin_time': 'Time', 'Procedure_name': 'Text'})
# Drop na 'Time' values
medications = medications[~medications['Time'].isna()]

In [21]:
# Save medications to parquet
medications.to_parquet(os.path.join(intermediate_dir, 'medications.parquet'), engine='pyarrow')

## Concatenating everything

In [113]:
# Read the medications parquet file
medications = pd.read_parquet(os.path.join(intermediate_dir, 'medications.parquet'), engine='pyarrow')
# Read the labs parquet file
labs = pd.read_parquet(os.path.join(intermediate_dir, 'labs.parquet'), engine='pyarrow')
# Read the vitals parquet file
vitals_nofreq = pd.read_parquet(os.path.join(intermediate_dir, 'vitals_nofreq.parquet'), engine='pyarrow')
# Read the vitals_freq parquet file
vitals_freq = pd.read_parquet(os.path.join(intermediate_dir, 'vitals_freq.parquet'), engine='pyarrow')

In [118]:
# Create 'Text' column in vitals_freq: measure: value
vitals_freq['Text'] = vitals_freq['Measure'] + ': ' + vitals_freq['Value'].astype(str) 
# Drop the Measure column
vitals_freq = vitals_freq[['CSN', 'Text', 'Time']]

In [119]:
vitals_freq.head()

,CSN,Text,Time
0,99314694,HR: 81.0,2231-09-14T13:44:37Z
1,99378047,HR: 62.0,2217-07-18T18:49:54Z
2,99998773,HR: 140.0,2235-07-25T00:21:40Z
3,99998773,HR: 108.0,2235-07-25T04:10:40Z
4,99050197,HR: 55.0,2249-07-09T08:12:13Z


In [120]:
# Concatenate the four dataframes
combined = pd.concat([labs, vitals_freq, vitals_nofreq, medications], ignore_index=True)
# Sort by CSN and Time
combined = combined.sort_values(by=['CSN', 'Time'])

In [121]:
# Vectorized string replacement using string accessor (.str)
combined['Time_2000'] = '2000' + combined['Time'].str[4:]
# Convert the modified strings to datetime objects
combined['Time_2000_dt'] = pd.to_datetime(combined['Time_2000'], infer_datetime_format=True, errors='coerce')
# Drop the original 'Time_2000' column 
combined = combined.drop(columns=['Time_2000'])

/tmp/ipykernel_914260/972886350.py:4: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  combined['Time_2000_dt'] = pd.to_datetime(combined['Time_2000'], infer_datetime_format=True, errors='coerce')


In [122]:
# Remove na values in 'Time_2000_dt'
combined = combined[~combined['Time_2000_dt'].isna()]
# Drop Time column 
combined = combined.drop(columns=['Time'])
# Rename 'Time_2000_dt' to 'Time'
combined = combined.rename(columns={'Time_2000_dt': 'Time'})

In [123]:
# drop all columns except CSN, Text and Time
combined = combined[['CSN', 'Text', 'Time']]

In [14]:
combined = pd.read_parquet(os.path.join("intermediate", 'combined.parquet'), engine='pyarrow')

In [78]:
len(combined)

46239387

In [125]:
# count number of unique CSNs
unique_csns = combined['CSN'].nunique()
print(f"Number of unique CSNs: {unique_csns}")

Number of unique CSNs: 117025


## Visits

In [17]:
# load visits 
visits = pd.read_csv(mc_med_path + 'visits.csv', dtype={'CSN': 'Int64'})

In [18]:
# count number of unique CSNs
unique_csns_visits = visits['CSN'].nunique()
print(f"Number of unique CSNs in visits: {unique_csns_visits}")

Number of unique CSNs in visits: 118385


In [19]:
# Drop na values in Admit_time
visits = visits[~visits['Arrival_time'].isna()]
# now let's convert the Admit_time to datetime using the same process as above
visits['Arrival_time_2000'] = '2000' + visits['Arrival_time'].astype(str).str[4:]
# Convert the modified strings to datetime objects
visits['Arrival_time_2000_dt'] = pd.to_datetime(visits['Arrival_time_2000'])

In [20]:
# keep CSN and Admit_time_2000_dt
# visits = visits[['CSN', 'Arrival_time_2000_dt']]
# Rename 'Admit_time_2000_dt' to 'Time'
visits = visits.rename(columns={'Arrival_time_2000_dt': 'Arrival_time'})

In [21]:
# Merge the two dataframes based on the common 'CSN' column
merged_df = pd.merge(combined, visits, on='CSN', how='inner')

# Calculate the time difference in minutes
merged_df['TimeDifferenceInMinutes'] = (merged_df['Time'] - merged_df['Arrival_time']).dt.total_seconds() / 60

TypeError: '<' not supported between instances of 'str' and 'int'

In [ ]:
merged_df.head()

In [138]:
# Keep just CSN, Text and TimeDifferenceInMinutes
merged_df = merged_df[['CSN', 'Text', 'TimeDifferenceInMinutes']]

In [ ]:
# Save the merged dataframe to parquet
merged_df.to_parquet(os.path.join(intermediate_dir, 'final_df.parquet'), engine='pyarrow', index=False)

Number of unique CSNs in merged_df: 117025


In [ ]:
# number of unique CSNs in merged_df
unique_csns_merged = merged_df['CSN'].nunique()
print(f"Number of unique CSNs in merged_df: {unique_csns_merged}")

In [4]:
final_df = pd.read_parquet(os.path.join("intermediate", 'final_df.parquet'), engine='pyarrow')

In [6]:
# sort by CSN and TimeDifferenceInMinutes
final_df = final_df.sort_values(by=['CSN', 'TimeDifferenceInMinutes'])
final_df.head()

,CSN,Text,TimeDifferenceInMinutes
0,98816159,RED BLOOD CELLS (RBC): 3.59 MIL/uL,158.0
1,98816159,MEAN CORPUSCULAR VOLUME (MCV): 88.6 fL,158.0
2,98816159,HEMOGLOBIN (HGB): 10.0 g/dL,158.0
3,98816159,WHITE BLOOD CELLS (WBC): 10.0 K/uL,158.0
4,98816159,MEAN CORPUSCULAR HEMOGLOBIN CONCENTRATION (MCH...,158.0


In [ ]:
# load visits
visits = pd.read_csv(mc_med_path + 'visits.csv', dtype={'CSN': 'Int64'})

## Final

In [ ]:
DATA_DIR = 'data'
labels = pd.read_parquet(our_mcmed + 'eSOFA.parquet')

In [10]:
pos_labels = labels[labels['Label'] == True].copy()
print(f"Number of positive labels: {len(pos_labels)}")
print("Total number of labels: ", len(labels))

Number of positive labels: 1738
Total number of labels:  15027


In [17]:
# Find number of pos_labels where Trigger_time > 2 (don't convert to minutes)
pos_labels_2h = pos_labels[pos_labels['Trigger_time'] > 2].copy()
print(f"Number of positive labels after filtering for 2 hrs: {len(pos_labels_2h)}")

Number of positive labels after filtering for 2 hrs: 311


In [18]:
# randomly sample 3000 negative labels 
neg_labels = labels[labels['Label'] == False].copy()
neg_labels = neg_labels.sample(n=3000, random_state=42)

In [20]:
# Concatenate the positive and negative labels
labels_2h = pd.concat([pos_labels_2h, neg_labels], ignore_index=True)
# Get the CSNs from the labels_2h dataframe
csns_2h = labels_2h['CSN'].unique()
print(f"Number of unique CSNs in labels_2h: {len(csns_2h)}")

Number of unique CSNs in labels_2h: 3311


In [23]:
# load final df
final_df = pd.read_parquet(os.path.join("intermediate", 'final_df.parquet'), engine='pyarrow')
# Filter the final_df dataframe to only include the CSNs in labels_2h
final_df_2h = final_df[final_df['CSN'].isin(csns_2h)].copy()

In [25]:
# only keep items where TimeDifferenceInMinutes is less than 120
final_df_2h = final_df_2h[final_df_2h['TimeDifferenceInMinutes'] < 120].copy()

In [ ]:
# Sort final_df_2h by CSN and TimeDifferenceInMinutes
final_df_2h = final_df_2h.sort_values(by=['CSN', 'TimeDifferenceInMinutes'])


,CSN,Text,TimeDifferenceInMinutes
419,98816643,"BILIRUBIN, TOTAL: .4 mg/dL",78.0
420,98816643,CHLORIDE: 97 mmol/L,78.0
421,98816643,CO2: 24 mmol/L,78.0
422,98816643,ALT (SGPT): 27 U/L,78.0
423,98816643,CORRECTED CALCIUM: 9.2 mg/dL,78.0


In [37]:
# for each CSN, condense the rows into a single row with a string of all the Text values, in the format Time: Text
def condense_rows(group):
    # Create a string of all the Text values, in the format Time: Text
    condensed_text = ', '.join(f"{row['TimeDifferenceInMinutes']}: {row['Text']}" for _, row in group.iterrows())
    # Return a single row with the CSN and the condensed text
    return pd.Series({'CSN': group['CSN'].iloc[0], 'Condensed_Text': condensed_text})
# Group by CSN and apply the condense_rows function
condensed_df = final_df_2h.groupby('CSN').apply(condense_rows).reset_index(drop=True)

In [42]:
# load visits
visits = pd.read_csv(mc_med_path + 'visits.csv', dtype={'CSN': 'Int64'})

In [43]:
# get the visits in csns_2h
visits_2h = visits[visits['CSN'].isin(csns_2h)].copy()

In [47]:
# add a new column: 'Condensed_Text' to visits_2h; which combines the Age, Gender, Race, Means_of_arrival and Triage_<lab> columns into a single string
def condense_visits(group):
    chosen_cols = ['Age', 'Gender', 'Race', 'Means_of_arrival', 'Triage_Temp', 'Triage_HR', 'Triage_RR', 'Triage_SBP', 'Triage_DBP', 'Triage_SPO2', 'Triage_acuity', 'CC']
    # Create a string of all the Text values, in the format Time: Text
    condensed_text = ', '.join(f"{col}: {group[col].iloc[0]}" for col in group.columns if col in chosen_cols)
    # Return a single row with the CSN and the condensed text
    return pd.Series({'CSN': group['CSN'].iloc[0], 'Condensed_Text': condensed_text})
# Group by CSN and apply the condense_rows function
condensed_visits_df = visits_2h.groupby('CSN').apply(condense_visits).reset_index(drop=True)

In [48]:
condensed_visits_df.head()

,CSN,Condensed_Text
0,98816643,"Age: 50, Gender: F, Race: Native Hawaiian or O..."
1,98819977,"Age: 62, Gender: F, Race: Other, Means_of_arri..."
2,98821846,"Age: 54, Gender: F, Race: Asian, Means_of_arri..."
3,98824756,"Age: 78, Gender: M, Race: Other, Means_of_arri..."
4,98828108,"Age: 60, Gender: F, Race: White, Means_of_arri..."


In [51]:
# Now, let's merge the two dataframes based on the common 'CSN' column
merged_condensed_df = pd.merge(condensed_df, condensed_visits_df, on='CSN', how='inner')
# Rename the columns
merged_condensed_df = merged_condensed_df.rename(columns={'Condensed_Text_x': 'Condensed_Text_Labs_Vitals_Meds', 'Condensed_Text_y': 'Condensed_Text_Visits'})
# Combine the two condensed text columns into a single column with a separator
merged_condensed_df['Combined_Condensed_Text'] = 'Patient Info\n=====\n' + merged_condensed_df['Condensed_Text_Labs_Vitals_Meds'] + '\n===== Labs, Vitals, Medications\n=====' + merged_condensed_df['Condensed_Text_Visits']

In [ ]:
# merge with labels_2h
labels_2h = labels_2h.rename(columns={'Trigger_time': 'Trigger_time'})
# Merge the two dataframes based on the common 'CSN' column
merged_labels_df = pd.merge(merged_condensed_df, labels_2h, on='CSN', how='inner')

In [55]:
# retain just the CSN, Combined_Condensed_Text, Label, and Trigger_time columns 
merged_labels_df = merged_labels_df[['CSN', 'Combined_Condensed_Text', 'Label', 'Trigger_time']]
# Save the merged dataframe to parquet
merged_labels_df.to_parquet(os.path.join('merged_labels_df.parquet'), engine='pyarrow', index=False)

In [ ]:
# count number of True labels (sanity check)
num_true_labels = merged_labels_df['Label'].sum()
print(f"Number of True labels: {num_true_labels}")

Number of True labels: 306
